# Глава 6. Честная проверка: train/test, baseline и утечка данных

_Ноутбук сгенерирован из учебника «Машинное обучение для начинающих». Запускайте ячейки по порядку сверху вниз в Google Colab._

**Цель главы  🎯**

После этой главы ты сможешь разделить данные на train и test и объяснить зачем, построить baseline-модель, распознать утечку данных и привести пример, а также честно оценить модель через accuracy.

Это, возможно, самая важная глава всего курса. Здесь речь не о новом алгоритме, а о том, как понять — хороша модель или мы себя обманываем. Без этого навыка любой красивый результат ничего не стоит. Именно на этих ошибках спотыкается большинство новичков, и именно их умение избегать отличает настоящего специалиста.

## Разделение на train и test

**Аналогия: зазубрил или понял  💡**

Представь ученика, который выучил ответы к конкретным задачам наизусть. На этих же задачах он получит 100%. Но дай ему новую задачу — и он провалится. Он зазубрил, а не понял. Модель ведёт себя точно так же: если проверять её на тех же данных, на которых она училась, она покажет отличный результат, но это ничего не говорит о том, как она справится с новыми, реальными данными.

Поэтому данные всегда делят на две части. Обучающая выборка (train) — на ней модель учится, видит правильные ответы. Тестовая выборка (test) — её модель никогда не видит при обучении, и именно на ней мы честно проверяем качество, имитируя встречу с новыми данными. Обычно берут 80% на обучение и 20% на тест.

In [ ]:
# === Подготовка: выполни первой, если открыл новый блокнот ===
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
df = pd.read_csv(url).dropna(subset=["bill_length_mm", "body_mass_g", "species"])
X = df[["bill_length_mm", "body_mass_g"]]
y = df["species"]

from sklearn.model_selection import train_test_split

# test_size=0.2 → 20% данных уходит в тест
# random_state=42 → фиксируем случайность, чтобы результат повторялся
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train, y_train)          # учимся ТОЛЬКО на train

from sklearn.metrics import accuracy_score
pred = model.predict(X_test)          # проверяем на невиданном test
print("Accuracy:", accuracy_score(y_test, pred))

_Ожидаемый результат: доля правильных ответов на тесте, например 0.68._

**Внимание, частая ошибка  ⚠️**

Проверять модель на обучающих данных — ошибка номер один у новичков. Результат будет завышенным и совершенно бесполезным. Правило железное: тест модель не видит до самого конца. И всегда ставь random_state — иначе при каждом запуске разбиение будет разным, результаты не сравнить, и ты не поймёшь, стало лучше или просто повезло с разбиением.

## Baseline: с чем вообще сравнивать?

Допустим, модель дала accuracy 0,68. Это хорошо или плохо? Ответить невозможно, пока нет точки отсчёта. Такой точкой служит baseline — самая простая, «тупая» модель, какую можно придумать. Например: «всегда предсказывай самый частый класс», не глядя на признаки вообще.

Если самый частый вид пингвинов (Adelie) составляет около половины данных, то модель, которая тупо всем ставит «Adelie», получит accuracy примерно 0,50 — просто угадывая большинство. Значит, наши ~0,75 — это заметный плюс к тупому угадыванию. Baseline мгновенно показывает, стоит ли модель вообще чего-то или она едва лучше монетки.

In [ ]:
from sklearn.dummy import DummyClassifier

# "most_frequent" — всегда предсказывает самый частый класс
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
print("Baseline accuracy:", baseline.score(X_test, y_test))

**Полезно знать  ℹ️**

Baseline — это не формальность, а компас. Если твоя сложная модель едва обгоняет baseline, значит, либо данные не содержат полезного сигнала, либо ты что-то делаешь не так. А если модель ХУЖЕ baseline — это явный признак ошибки. Всегда строй baseline первым, до всякой умной модели.

## Data leakage — коварная утечка данных

**Аналогия: подсмотреть ответы  💡**

Утечка данных (data leakage) — это когда модель случайно «подсматривает» в тестовые данные или в информацию, которой в реальный момент предсказания ещё не будет. Это как если бы ученик готовился к экзамену по украденным заранее ответам: на экзамене он блеснёт, а реальных знаний нет. В жизни такая модель с треском провалится.

Классический пример утечки: мы предсказываем, вернёт ли клиент кредит, и случайно оставляем в признаках колонку «сумма штрафа за невозврат». Но штраф появляется только ПОСЛЕ невозврата! В реальный момент, когда мы решаем, давать ли кредит, этой информации ещё не существует. Модель покажет фантастическую точность на исторических данных, но в жизни окажется бесполезной, потому что подсматривала в будущее.

Утечка коварна тем, что даёт отличные метрики — и потому её легко не заметить и обрадоваться. Правило: спрашивай про каждый признак — «а будет ли эта информация доступна в реальный момент предсказания?». Если нет — это утечка.

**Полезно знать  ℹ️**

Ещё один частый источник утечки — масштабировать признаки или заполнять пропуски на ВСЕХ данных сразу, до разделения на train/test. Тогда информация из теста (например, среднее значение) «протекает» в обучение. Именно поэтому в главе 10 мы соберём Pipeline: он делает все преобразования правильно, только на train, и защищает от этой ошибки автоматически.

## Соревнование: честный подбор K

Теперь мы умеем проверять честно — и можем вернуться к вопросу из прошлой главы: как выбрать K? Попробуем разные значения и найдём лучшее, но оценивать будем ТОЛЬКО по тесту, а не по обучению.

In [ ]:
for k in [1, 3, 5, 7, 15, 30]:
    m = KNeighborsClassifier(n_neighbors=k)
    m.fit(X_train, y_train)
    train_acc = m.score(X_train, y_train)
    test_acc = m.score(X_test, y_test)
    print(f"K={k:2d}: train={train_acc:.3f}, test={test_acc:.3f}")

Внимательно посмотри на две колонки. При K=1 train почти идеальный (модель запомнила каждую точку), но test заметно ниже — это первый звоночек переобучения, о котором подробно в следующей главе. При большом K обе метрики падают — модель слишком груба. Где-то посередине — золотая середина. Именно поэтому смотреть надо на test, а не на train.

## Задания

Базовый уровень

Запусти цикл подбора K. Построй мысленно (или на бумаге) график: по горизонтали K, по вертикали test-accuracy. При каком K точность на тесте максимальна? Это и есть твой лучший K.

Со звёздочкой  ⭐

Придумай два своих примера утечки данных для реальных задач (например, предсказание оценки за экзамен, предсказание, купит ли человек товар). В каждом укажи, какой признак «подсматривает в будущее» и почему в реальности его не будет.

Вопросы для обсуждения  💬

Зачем вообще делить данные, если можно обучить модель на всех данных сразу — так ведь больше примеров?

Модель даёт accuracy 0,55, а baseline — 0,60. Что это говорит о модели?

Почему утечка данных особенно опасна — ведь метрики при ней хорошие?

**Проверь себя  ℹ️**

1. Зачем делить данные на train и test?

2. Что такое baseline и зачем он нужен?

3. Приведи свой пример утечки данных.